# Research notebook




In [ ]:
# Portable project paths; this cell does not load a model.
from pathlib import Path
import sys

_candidates = (Path.cwd(), *Path.cwd().parents)
_project_root = next((p for p in _candidates if (p / "project_paths.py").is_file()), None)
if _project_root is None:
    raise RuntimeError("Start Jupyter from the retrieval repository or one of its subdirectories.")
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))
from project_paths import (GALLERY_DIR, SIGLIP_MODEL, REFERENCE_CLASS_FOLDERS,
                           reference_class_indices)
# Experiment settings. Edit REFERENCE_CLASS_FOLDERS in project_paths.py for your data.
target_classes = list(REFERENCE_CLASS_FOLDERS)
CLIP_MODEL = "ViT-B/32"
TARGET_CLASS = target_classes[-1]  # Default: Porcelain
REFERENCE_COUNT = 50
N_CLUSTERS = 4
REFERENCE_BUDGET = 20
ELBOW_MAX_K = 10



In [1]:
import os
from PIL import Image
import torch
import clip
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load(CLIP_MODEL, device=device)
import numpy as np
from pathlib import Path

from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset
import random
from transformers import BertTokenizer, BertForSequenceClassification, CLIPProcessor, CLIPModel
from tqdm import tqdm
from transformers import AutoProcessor, AutoModel, BitsAndBytesConfig
import accelerate
import sentencepiece
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [2]:
#区分数据集为参考图和图片池
def split(dataset, target_label, ref_number, seed=0):
    #获取所有图片索引，在目标类别选取参考图，再分开数据集
    all_indices=list(range(len(dataset)))
    target_indices=[idx for idx in all_indices if dataset[idx][1]==target_label]
    random.seed(seed)
    ref_indices=random.sample(target_indices, ref_number)
    gallery_indices=[idx for idx in all_indices if idx not in ref_indices]
    return Subset(dataset,ref_indices), Subset(dataset,gallery_indices)
def f1_score(similarity,gallery_label,threshold,target_label):
    #pos是布尔值，在simliarity中使用可以获得目标标签的变量
    pos_bool=(gallery_label==target_label)
    neg_bool=(gallery_label!=target_label)
    pos=similarity[pos_bool]
    neg=similarity[neg_bool]
    TP=sum(pos>=threshold)
    FP=sum(neg>=threshold)
    FN=sum(pos<threshold)
    precision=TP/(TP+FP+0.0001)
    recall=TP/(TP+FN+0.0001)
    f1=2*precision*recall/(precision+recall+0.0001)
    return f1,precision,recall
def find_threshold(similarity,gallery_label,target_label):
    thresholds=np.linspace(np.min(similarity),np.max(similarity),100)
    best_f1=0
    best_precision=0
    best_recall=0
    for threshold in thresholds:
        f1,precision,recall=f1_score(similarity,gallery_label,threshold,target_label)
        if f1 > best_f1:
            best_f1=f1
            best_precision=precision
            best_recall=recall
    return best_f1,best_precision,best_recall


In [3]:
# Experiment settings are in the setup cell.


In [4]:
model = AutoModel.from_pretrained(SIGLIP_MODEL).to(device).eval()
processor = AutoProcessor.from_pretrained(SIGLIP_MODEL)
def clip_transform(image):
    # 1. 用processor处理图像，得到字典
        inputs = processor(images=image, return_tensors="pt")
    # 2. 提取pixel_values并移除batch维度（从[1,3,224,224]转为[3,224,224]）
        pixel_values = inputs["pixel_values"].squeeze(0)
        return pixel_values
dataset=ImageFolder(root=str(GALLERY_DIR),transform=clip_transform)
print('数据集加载完成')

数据集加载完成


In [9]:
dic = reference_class_indices(dataset.class_to_idx)
target_class=TARGET_CLASS
ref_number=REFERENCE_COUNT
target_label=dic[target_class]
 #使用imageFolder导入数据集
ref_dataset,gallery_dataset=split(dataset,target_label,ref_number)
ref = torch.utils.data.DataLoader(ref_dataset,shuffle=False)
gallery=torch.utils.data.DataLoader(gallery_dataset, batch_size=256, num_workers=0, shuffle=False)
print('数据集创建完成')
model.eval()
gallery_features=[]
ref_features=[]
with torch.no_grad():
    for pixel_values, _ in gallery:  # 直接获取pixel_values（已由clip_transform处
                pixel_values = pixel_values.to(device)
                batch_features = model.get_image_features(pixel_values=pixel_values)  # 正确参数
                batch_features = batch_features / batch_features.norm(dim=1, keepdim=True)
                gallery_features.append(batch_features)
    print('特征创建完成')
    for pixel_values, _ in ref:
                pixel_values = pixel_values.to(device)
                batch_features = model.get_image_features(pixel_values=pixel_values)  # 正确参数
                batch_features = batch_features / batch_features.norm(dim=1, keepdim=True)
                ref_features.append(batch_features)
    print('特征创建完成')
    # 合并所有批次的特征
gallery_features = torch.cat(gallery_features, dim=0)

数据集创建完成
特征创建完成
特征创建完成


In [10]:
#k-means聚类
#手肘法确定k
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
X=[]
for i in range(len(ref_features)):
    X.append(ref_features[i][0].cpu().numpy())
# 生成示例数据
# 计算不同K值下的惯性（inertia，样本到其所属聚类中心的距离平方和）
inertia = []
for k in range(1, ELBOW_MAX_K + 1):
    kmeans = KMeans(n_clusters=k, random_state=0).fit(X)
    inertia.append(kmeans.inertia_)
# 绘制手肘图
plt.plot(range(1, ELBOW_MAX_K + 1), inertia, marker='o')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method For Optimal k')
plt.show()

In [11]:

features=X
features=np.array(features)
k=N_CLUSTERS
kmeans = KMeans(n_clusters=k, random_state=0).fit(features)
distances = np.zeros(len(features))
labels=kmeans.labels_.astype(int)
for i in range(k):
    cluster_mask = (labels == i)
    print(cluster_mask)
    cluster_features = features[cluster_mask,:]
    center = kmeans.cluster_centers_[i].reshape(1, -1)
    distances[cluster_mask] = np.linalg.norm(cluster_features - center, axis=1)

# 按聚类大小比例分配20个名额
cluster_sizes = np.bincount(kmeans.labels_)  # 每个聚类的样本数
total_samples = len(features)

# 计算初始分配名额（向下取整）
initial_quota = np.floor(cluster_sizes * REFERENCE_BUDGET / total_samples).astype(int)
remaining = REFERENCE_BUDGET - initial_quota.sum()  # 剩余名额

# 按聚类大小降序排列，分配剩余名额
if remaining > 0:
    sorted_indices = np.argsort(-cluster_sizes)  # 负号表示降序
    for i in sorted_indices:
        if remaining <= 0:
            break
        initial_quota[i] += 1
        remaining -= 1

# 从每个聚类中选择距离中心最近的样本
selected_indices = []
for i in range(k):
    cluster_mask = (kmeans.labels_ == i)
    cluster_indices = np.where(cluster_mask)[0]

    # 如果该聚类有分配名额，则选择前n个最近的样本
    if initial_quota[i] > 0:
        cluster_distances = distances[cluster_mask]
        top_indices = cluster_indices[np.argsort(cluster_distances)[:initial_quota[i]]]
        selected_indices.extend(top_indices)

# 提取选中的20个点的特征
selected_features = features[selected_indices]

print(f"选中的{REFERENCE_BUDGET}个点的索引: {selected_indices}")
print(f"各聚类分配的名额: {initial_quota}")
print(f"选中特征的形状: {selected_features.shape}")  # 应输出 (20, 512)

[ True  True False False False  True False False False False False False
 False  True False False False False  True False False False  True False
 False False False False False  True False False  True False False False
  True False  True False  True False False False False False False False
 False False]
[False False False False False False False False  True False False False
 False False False False False False False False False False False False
 False False False False  True False  True False False False False False
 False False False False False False False False False False False False
 False False]
[False False False False  True False  True False False False  True  True
  True False  True False  True False False  True False  True False  True
  True  True  True  True False False False False False  True  True False
 False False False False False False  True  True False  True False  True
  True False]
[False False  True  True False False False  True False  True False False
 False Fa

In [1]:

selected_features_mean = selected_features.mean(axis=0, keepdims=True)
selected_features_tensor = torch.from_numpy(selected_features_mean).float().to(device)
ref_features=selected_features_tensor
print(selected_features_tensor)
#计算相似度矩阵
similarity = gallery_features @ ref_features.T
similarity=  similarity.cpu().numpy()
gallery_label = [sample[1] for sample in gallery_dataset]  # sample[1]是标签
gallery_label = np.array(gallery_label)
best_f1,best_precision,best_recall=find_threshold(similarity,gallery_label,target_label)
print(f'{target_class},{ref_number}:f1:{best_f1},precision:{best_precision},recall:{best_recall}')